In [ ]:
import os
os.chdir(os.path.join('..', 'results'))

# Strategy 1 — 30 Repeated Runs (PartB: New Datasets, 8 configs)
**Seeds 0–29.**

# Strategy 1 — Part B: New Datasets (8 configurations)

## SECTION 1: ORIGINAL 5 DATASET LOADERS (unchanged)

In [4]:
import numpy as np
import pandas as pd
import pickle
import requests
import io
from scipy.io import arff
from sklearn.cluster import KMeans
# from sklearn.preprocessing import StandardScaler, LabelEncoder  # ORIGINAL (Reviewer 3, Comment 8)
from sklearn.preprocessing import StandardScaler, LabelEncoder, OneHotEncoder  # FIXED: added OneHotEncoder (LabelEncoder import kept, unused elsewhere)
from sklearn.metrics import (
    silhouette_score, silhouette_samples, calinski_harabasz_score,
    davies_bouldin_score, adjusted_rand_score, normalized_mutual_info_score,
    adjusted_mutual_info_score, fowlkes_mallows_score, v_measure_score
)
from scipy.spatial.distance import cdist, pdist
from scipy.stats import entropy
import matplotlib.pyplot as plt
import seaborn as sns
from collections import defaultdict, Counter
import warnings
warnings.filterwarnings('ignore')
np.random.seed(0)

In [5]:
# =============================================================================
# SECTION 1: ORIGINAL 5 DATASET LOADERS (unchanged)
# =============================================================================

def load_adult():
    print("\n" + "="*80)
    print("Loading Adult Income Dataset")
    print("="*80)
    columns = [
        'age', 'workclass', 'fnlwgt', 'education', 'education-num',
        'marital-status', 'occupation', 'relationship', 'race', 'sex',
        'capital-gain', 'capital-loss', 'hours-per-week', 'native-country', 'income'
    ]
    adult = pd.read_csv("../data/adult/adult.data", names=columns, header=None)
    adult = adult.replace(' ?', np.nan).dropna()
    adult['sex_binary'] = (adult['sex'].str.strip() == 'Male').astype(int)
    adult_features = [c for c in adult.columns if c not in ["income", "sex", "race", "sex_binary"]]
    print(f"Shape: {adult.shape}")
    print(f"Gender: 0(F)={( adult['sex_binary']==0).sum()}, 1(M)={(adult['sex_binary']==1).sum()}")
    return adult, adult_features, 'sex_binary'


def load_compas():
    print("\n" + "="*80)
    print("Loading COMPAS Dataset")
    print("="*80)
    compas = pd.read_csv("../data/compas/compas-scores.csv")
    compas = compas[
        (compas['days_b_screening_arrest'] <= 30) &
        (compas['days_b_screening_arrest'] >= -30) &
        (compas['is_recid'] != -1)
    ]
    selected_cols = compas.columns[compas.isnull().sum() == 0].tolist()
    compas = compas[selected_cols]
    compas['sex_binary'] = (compas['sex'] == 'Male').astype(int)
    compas_features = [c for c in compas.columns if c not in ["is_recid", "sex", "race", "sex_binary"]]
    print(f"Shape: {compas.shape}")
    print(f"Gender: 0(F)={(compas['sex_binary']==0).sum()}, 1(M)={(compas['sex_binary']==1).sum()}")
    return compas, compas_features, 'sex_binary'


def load_german():
    print("\n" + "="*80)
    print("Loading German Credit Dataset")
    print("="*80)
    german_columns = [
        'checking_status', 'duration', 'credit_history', 'purpose', 'credit_amount',
        'savings_status', 'employment', 'installment_rate', 'personal_status_sex',
        'other_parties', 'residence_since', 'property_magnitude', 'age',
        'other_payment_plans', 'housing', 'existing_credits', 'job',
        'num_dependents', 'own_telephone', 'foreign_worker', 'class'
    ]
    german = pd.read_csv("../data/german/german.data", sep=' ', names=german_columns, header=None)
    def extract_sex(s):
        return 'male' if s in ['A91', 'A93', 'A94'] else ('female' if s in ['A92', 'A95'] else 'unknown')
    german['sex'] = german['personal_status_sex'].apply(extract_sex)
    german = german[german['sex'] != 'unknown']
    german['sex_binary'] = (german['sex'] == 'male').astype(int)
    german_features = [c for c in german.columns if c not in ["class", "sex", "sex_binary"]]
    print(f"Shape: {german.shape}")
    print(f"Gender: 0(F)={(german['sex_binary']==0).sum()}, 1(M)={(german['sex_binary']==1).sum()}")
    return german, german_features, 'sex_binary'


def load_credit():
    print("\n" + "="*80)
    print("Loading Default Credit Card Dataset")
    print("="*80)
    credit = pd.read_csv("../data/default of credit card clients/default of credit card clients.csv", header=0)
    if 'X1' in credit.columns or credit.columns[0] == 'ID':
        credit.columns = [
            'ID', 'LIMIT_BAL', 'SEX', 'EDUCATION', 'MARRIAGE', 'AGE',
            'PAY_0', 'PAY_2', 'PAY_3', 'PAY_4', 'PAY_5', 'PAY_6',
            'BILL_AMT1', 'BILL_AMT2', 'BILL_AMT3', 'BILL_AMT4', 'BILL_AMT5', 'BILL_AMT6',
            'PAY_AMT1', 'PAY_AMT2', 'PAY_AMT3', 'PAY_AMT4', 'PAY_AMT5', 'PAY_AMT6', 'default'
        ]
    credit = credit.drop(columns=['ID'])
    credit['SEX'] = pd.to_numeric(credit['SEX'], errors='coerce')
    credit['sex'] = credit['SEX'].map({1.0: 'male', 2.0: 'female'})
    credit = credit.dropna(subset=['sex'])
    credit['sex_binary'] = (credit['sex'] == 'male').astype(int)
    credit_features = [c for c in credit.columns if c not in ["default", "sex", "SEX", "sex_binary"]]
    print(f"Shape: {credit.shape}")
    print(f"Gender: 0(F)={(credit['sex_binary']==0).sum()}, 1(M)={(credit['sex_binary']==1).sum()}")
    return credit, credit_features, 'sex_binary'


def load_law():
    print("\n" + "="*80)
    print("Loading Law School (LSAC) Dataset")
    print("="*80)
    lsac = pd.read_csv("../data/law/law_dataset.csv")
    lsac['sex'] = lsac['male'].map({1: 'Male', 0: 'Female', 1.0: 'Male', 0.0: 'Female'})
    lsac = lsac[lsac['sex'].notna()]
    lsac['sex_binary'] = (lsac['sex'] == 'Male').astype(int)
    drop_cols = ['male', 'sex', 'pass_bar', 'sex_binary', 'racetxt'] 
    lsac_features = [c for c in lsac.columns if c not in drop_cols]
    print(f"Shape: {lsac.shape}")
    print(f"Gender: 0(F)={(lsac['sex_binary']==0).sum()}, 1(M)={(lsac['sex_binary']==1).sum()}")
    return lsac, lsac_features, 'sex_binary'

# =============================================================================
# RACE & COMBINED LOADERS — Adult, COMPAS, Law
# German, Credit: gender only
# =============================================================================

def load_adult_race():
    columns = [
        'age', 'workclass', 'fnlwgt', 'education', 'education-num',
        'marital-status', 'occupation', 'relationship', 'race', 'sex',
        'capital-gain', 'capital-loss', 'hours-per-week', 'native-country', 'income'
    ]
    df = pd.read_csv("../data/adult/adult.data", names=columns, header=None)
    df = df.replace(' ?', np.nan).dropna()
    df['race'] = df['race'].str.strip()
    df['race_binary'] = (df['race'] == 'White').astype(int)
    features = [c for c in df.columns if c not in ["income", "sex", "race", "race_binary"]]
    print(f"Adult (race) | 0(NonWhite)={(df['race_binary']==0).sum()}, 1(White)={(df['race_binary']==1).sum()}")
    return df, features, 'race_binary'

def load_adult_combined():
    columns = [
        'age', 'workclass', 'fnlwgt', 'education', 'education-num',
        'marital-status', 'occupation', 'relationship', 'race', 'sex',
        'capital-gain', 'capital-loss', 'hours-per-week', 'native-country', 'income'
    ]
    df = pd.read_csv("../data/adult/adult.data", names=columns, header=None)
    df = df.replace(' ?', np.nan).dropna()
    df['race'] = df['race'].str.strip()
    df['sex_binary']  = (df['sex'].str.strip() == 'Male').astype(int)
    df['race_binary'] = (df['race'] == 'White').astype(int)
    df['combined']    = df['race_binary'] * 2 + df['sex_binary']
    features = [c for c in df.columns if c not in ["income","sex","race","sex_binary","race_binary","combined"]]
    print(f"Adult (combined):\n{df['combined'].value_counts().sort_index()}")
    print("0=NonWhite-Female, 1=NonWhite-Male, 2=White-Female, 3=White-Male")
    return df, features, 'combined'

def load_compas_race():
    df = pd.read_csv("../data/compas/compas-scores.csv")
    df = df[(df['days_b_screening_arrest']<=30)&(df['days_b_screening_arrest']>=-30)&(df['is_recid']!=-1)]
    df = df[df.columns[df.isnull().sum()==0].tolist()]
    df = df[df['race'].isin(['Caucasian','African-American'])].copy()
    df['race_binary'] = (df['race'] == 'African-American').astype(int)
    features = [c for c in df.columns if c not in ["is_recid","sex","race","race_binary"]]
    print(f"COMPAS (race) | 0(White)={(df['race_binary']==0).sum()}, 1(Black)={(df['race_binary']==1).sum()}")
    return df, features, 'race_binary'

def load_compas_combined():
    df = pd.read_csv("../data/compas/compas-scores.csv")
    df = df[(df['days_b_screening_arrest']<=30)&(df['days_b_screening_arrest']>=-30)&(df['is_recid']!=-1)]
    df = df[df.columns[df.isnull().sum()==0].tolist()]
    df = df[df['race'].isin(['Caucasian','African-American'])].copy()
    df['sex_binary']  = (df['sex'] == 'Male').astype(int)
    df['race_binary'] = (df['race'] == 'African-American').astype(int)
    df['combined']    = df['race_binary'] * 2 + df['sex_binary']
    features = [c for c in df.columns if c not in ["is_recid","sex","race","sex_binary","race_binary","combined"]]
    print(f"COMPAS (combined):\n{df['combined'].value_counts().sort_index()}")
    print("0=White-Female, 1=White-Male, 2=Black-Female, 3=Black-Male")
    return df, features, 'combined'

def load_law_race():
    df = pd.read_csv("../data/law/law_dataset.csv")
    df['sex'] = df['male'].map({1:'Male',0:'Female',1.0:'Male',0.0:'Female'})
    df = df[df['sex'].notna()]
    df['race_binary'] = df['racetxt'].astype(int)  # 1=White, 0=NonWhite
    features = [c for c in df.columns if c not in ['male','sex','pass_bar','racetxt','race_binary']]
    print(f"Law (race) | 0(NonWhite)={(df['race_binary']==0).sum()}, 1(White)={(df['race_binary']==1).sum()}")
    return df, features, 'race_binary'

def load_law_combined():
    df = pd.read_csv("../data/law/law_dataset.csv")
    df['sex'] = df['male'].map({1:'Male',0:'Female',1.0:'Male',0.0:'Female'})
    df = df[df['sex'].notna()]
    df['sex_binary']  = (df['sex'] == 'Male').astype(int)
    df['race_binary'] = df['racetxt'].astype(int)
    df['combined']    = df['race_binary'] * 2 + df['sex_binary']
    features = [c for c in df.columns if c not in ['male','sex','pass_bar','racetxt','sex_binary','race_binary','combined']]
    print(f"Law (combined):\n{df['combined'].value_counts().sort_index()}")
    print("0=NonWhite-Female, 1=NonWhite-Male, 2=White-Female, 3=White-Male")
    return df, features, 'combined'

## SECTION 2: NEW DATASET LOADERS — D6, D7, D8

In [7]:
import os
# SECTION 2: NEW DATASET LOADERS — D6, D7, D8
# Each supports mode='gender' | 'race' | 'combined'
# =============================================================================

def load_diabetes(mode='gender'):
    """
    D6: Diabetes 130-US Hospitals (UCI id=296)
    mode: 'gender' | 'race' | 'combined'
    race: 1=Caucasian, 0=AfricanAmerican
    combined: 0=Black-Female, 1=Black-Male, 2=White-Female, 3=White-Male
    """
    print("\n" + "="*80)
    print(f"Loading Diabetes 130-US Dataset (mode={mode})")
    print("="*80)
    from ucimlrepo import fetch_ucirepo
    ds = fetch_ucirepo(id=296)
    df = ds.data.features.copy()

    drop_cols = ['encounter_id', 'patient_nbr', 'weight', 'payer_code', 'medical_specialty']
    df = df.drop(columns=[c for c in drop_cols if c in df.columns])
    df = df[df['gender'].isin(['Male', 'Female'])]
    df = df[df['race'] != '?'].dropna(subset=['gender', 'race'])

    df['sex_binary'] = (df['gender'] == 'Male').astype(int)

    # For race/combined: keep only Caucasian and AfricanAmerican
    df_race = df[df['race'].isin(['Caucasian', 'AfricanAmerican'])].copy()
    df_race['race_binary'] = (df_race['race'] == 'Caucasian').astype(int)
    df_race['combined'] = df_race['race_binary'] * 2 + df_race['sex_binary']

    base_excl = ['gender', 'race', 'sex_binary', 'race_binary', 'combined']

    if mode == 'gender':
        features = [c for c in df.columns if c not in base_excl]
        print(f"Shape: {df.shape} | Gender: 0(F)={(df['sex_binary']==0).sum()}, 1(M)={(df['sex_binary']==1).sum()}")
        return df, features, 'sex_binary'
    elif mode == 'race':
        features = [c for c in df_race.columns if c not in base_excl]
        print(f"Shape: {df_race.shape} | Race: 0(Black)={(df_race['race_binary']==0).sum()}, 1(White)={(df_race['race_binary']==1).sum()}")
        return df_race, features, 'race_binary'
    elif mode == 'combined':
        features = [c for c in df_race.columns if c not in base_excl]
        print(f"Shape: {df_race.shape} | Combined:\n{df_race['combined'].value_counts().sort_index()}")
        print("0=Black-Female, 1=Black-Male, 2=White-Female, 3=White-Male")
        return df_race, features, 'combined'
    else:
        raise ValueError(f"mode must be 'gender'/'race'/'combined', got '{mode}'")


def load_dutch(mode='gender'):
    """
    D7: Dutch Census 2001
    mode: 'gender' only (no race attribute in this dataset)
    sex: 1=male, 2=female → sex_binary: 1=male, 0=female
    """
    print("\n" + "="*80)
    print(f"Loading Dutch Census Dataset (mode={mode})")
    print("="*80)
    if mode != 'gender':
        raise ValueError("Dutch Census only supports mode='gender' (no race attribute)")

    url = ('https://raw.githubusercontent.com/tailequy/fairness_dataset'
           '/main/Dutch_census/dutch_census_2001.arff')
    try:
        resp = requests.get(url, timeout=30)
        resp.raise_for_status()
        data, meta = arff.loadarff(io.StringIO(resp.text))
        df = pd.DataFrame(data)
        for col in df.select_dtypes(['object']).columns:
            df[col] = df[col].str.decode('utf-8')
        print(f"Loaded from URL. Shape: {df.shape}")
    except Exception as e:
        print(f"URL failed ({e}), trying local file ../data/dutch/dutch_census_2001.csv")
        df = pd.read_csv('../data/dutch/dutch_census_2001.csv')

    sex_col = [c for c in df.columns if c.lower() == 'sex'][0]
    unique_vals = set(df[sex_col].astype(str).unique())
    if unique_vals.issubset({'1', '2'}):
        df['sex_binary'] = (df[sex_col].astype(str) == '1').astype(int)
    else:
        df['sex_binary'] = (df[sex_col].str.lower().isin(['male', 'm', '1'])).astype(int)

    target_col = next((c for c in df.columns if c.lower() in ['occupation', 'label', 'class']), None)
    excl = [sex_col, 'sex_binary'] + ([target_col] if target_col else [])
    features = [c for c in df.columns if c not in excl]

    print(f"Gender: 0(F)={(df['sex_binary']==0).sum()}, 1(M)={(df['sex_binary']==1).sum()}")
    return df, features, 'sex_binary'


def load_meps(mode='gender'):
    print("\n" + "="*80)
    print(f"Loading MEPS Panel 20 (2016) Dataset (mode={mode})")
    print("="*80)

    import aif360
    meps_dir = os.path.join(os.path.dirname(aif360.__file__), 'data', 'raw', 'meps')
    df = pd.read_csv(meps_dir + r'/h192.csv', low_memory=False)
    print(f"Loaded h192.csv. Shape: {df.shape}")

    keep_cols = ['AGE31X','AGE42X','AGE53X','SEX','RACEV1X','MARRY31X','MARRY42X','MARRY53X',
                 'EDUCYR','HIDEG','FTSTU31X','FTSTU42X','FTSTU53X','ACTDTY31','ACTDTY42','ACTDTY53',
                 'HONRDC31','HONRDC42','HONRDC53','WRGFD31X','WRGFD42X','WRGFD53X',
                 'TTLP16X','POVCAT16','INSCOV16']
    available = [c for c in keep_cols if c in df.columns]
    print(f"Available key columns: {len(available)}/{len(keep_cols)}")
    df = df[available].dropna()
    print(f"Shape after dropna: {df.shape}")

    df['sex_binary']  = (df['SEX'] == 1).astype(int)
    df['race_binary'] = (df['RACEV1X'] == 1).astype(int)
    df['combined']    = df['race_binary'] * 2 + df['sex_binary']

    print(f"Sex distribution: {df['SEX'].value_counts().to_dict()}")
    print(f"Race distribution: {df['RACEV1X'].value_counts().to_dict()}")

    base_excl = ['SEX', 'RACEV1X', 'sex_binary', 'race_binary', 'combined']
    features = [c for c in df.columns if c not in base_excl]

    if mode == 'gender':
        print(f"Gender: 0(F)={(df['sex_binary']==0).sum()}, 1(M)={(df['sex_binary']==1).sum()}")
        return df, features, 'sex_binary'
    elif mode == 'race':
        print(f"Race: 0(NonWhite)={(df['race_binary']==0).sum()}, 1(White)={(df['race_binary']==1).sum()}")
        return df, features, 'race_binary'
    elif mode == 'combined':
        print(f"Combined:\n{df['combined'].value_counts().sort_index()}")
        print("0=NonWhite-Female, 1=NonWhite-Male, 2=White-Female, 3=White-Male")
        return df, features, 'combined'
    else:
        raise ValueError(f"mode must be gender/race/combined, got '{mode}'")

## SECTION 3: PREPROCESSING

In [9]:
def preprocess_dataset(df, feature_cols, sensitive_col, max_unique_ratio=0.5,
                        numeric_convert_threshold=0.95, datetime_convert_threshold=0.95):
    import re
    X_df = df[feature_cols].copy()
    n = len(X_df)
    id_name_pattern = re.compile(r'(name|first|last|dob)', re.IGNORECASE)

    num_cols = []
    cat_cols = []
    excluded_cols = []

    for col in feature_cols:
        if X_df[col].dtype != object:
            num_cols.append(col)
            continue

        # recover object columns that are actually numeric, mistyped as string
        converted_num = pd.to_numeric(X_df[col], errors='coerce')
        if converted_num.notna().mean() >= numeric_convert_threshold:
            X_df[col] = converted_num
            num_cols.append(col)
            continue

        # exclude known identifier style fields, name, first, last, dob, by column name
        if id_name_pattern.search(str(col)):
            excluded_cols.append(col)
            continue

        # exclude columns that are almost entirely parseable as dates, these are
        # not nominal categories and should not be One Hot Encoded as such
        converted_dt = pd.to_datetime(X_df[col], errors='coerce')
        if converted_dt.notna().mean() >= datetime_convert_threshold:
            excluded_cols.append(col)
            continue

        # exclude remaining high cardinality object columns, id, case number,
        # free text, and similar fields
        unique_ratio = X_df[col].nunique() / n
        if unique_ratio > max_unique_ratio:
            excluded_cols.append(col)
            continue

        cat_cols.append(col)

    # excluded_cols is intentionally not printed, inspect it directly if needed

    num_part = X_df[num_cols].values.astype(float) if num_cols else np.zeros((len(X_df), 0))
    if cat_cols:
        ohe = OneHotEncoder(sparse_output=False, handle_unknown='ignore')
        cat_part = ohe.fit_transform(X_df[cat_cols].astype(str))
    else:
        cat_part = np.zeros((len(X_df), 0))
    X = np.hstack([num_part, cat_part])
    if np.any(np.isnan(X)) or np.any(np.isinf(X)):
        X = np.nan_to_num(X, nan=0.0, posinf=0.0, neginf=0.0)
    X = StandardScaler().fit_transform(X)
    sensitive_attr = df[sensitive_col].values.astype(int)
    print(f"  Final shape: {X.shape}")
    print(f"  Sensitive groups: {dict(zip(*np.unique(sensitive_attr, return_counts=True)))}")
    return X, sensitive_attr


## SECTION 4: LOAD DATASETS

In [11]:
# =============================================================================
# SECTION 4: LOAD DATASETS — Part B (8 configurations)
# diabetes×3, dutch×1, meps×3
# =============================================================================

print("\n" + "="*80)
print("LOADING DATASETS — PART B")
print("="*80)
datasets = {}

# --- Diabetes: gender / race / combined ---
for mode in ['gender', 'race', 'combined']:
    name = f'diabetes_{mode}'
    try:
        df, features, sensitive_col = load_diabetes(mode=mode)
        X, sensitive = preprocess_dataset(df, features, sensitive_col)
        datasets[name] = {'X': X, 'sensitive': sensitive,
                          'n_samples': X.shape[0], 'n_features': X.shape[1]}
        print(f"✓ {name}: {X.shape}")
    except Exception as e:
        print(f"✗ {name}: {e}")

# --- Dutch Census: gender only ---
try:
    df, features, sensitive_col = load_dutch(mode='gender')
    X, sensitive = preprocess_dataset(df, features, sensitive_col)
    datasets['dutch_gender'] = {'X': X, 'sensitive': sensitive,
                                'n_samples': X.shape[0], 'n_features': X.shape[1]}
    print(f"✓ dutch_gender: {X.shape}")
except Exception as e:
    print(f"✗ dutch_gender: {e}")

# --- MEPS: gender / race / combined ---
for mode in ['gender', 'race', 'combined']:
    name = f'meps_{mode}'
    try:
        df, features, sensitive_col = load_meps(mode=mode)
        X, sensitive = preprocess_dataset(df, features, sensitive_col)
        datasets[name] = {'X': X, 'sensitive': sensitive,
                          'n_samples': X.shape[0], 'n_features': X.shape[1]}
        print(f"✓ {name}: {X.shape}")
    except Exception as e:
        print(f"✗ {name}: {e}")

print("\n" + "="*80)
print("DATASET SUMMARY")
print("="*80)
for name, data in datasets.items():
    print(f"  {name:30s}: {data['n_samples']:>7} samples, {data['n_features']:>4} features")

dataset_names = list(datasets.keys())
print(f"\nTotal configurations loaded: {len(dataset_names)}")
# Expected 8: diabetes×3, dutch×1, meps×3 (note: dutch has no race attribute)



LOADING DATASETS — PART B

Loading Diabetes 130-US Dataset (mode=gender)
Shape: (99492, 45) | Gender: 0(F)=53575, 1(M)=45917
  Final shape: (99492, 892)
  Sensitive groups: {0: 53575, 1: 45917}
✓ diabetes_gender: (99492, 892)

Loading Diabetes 130-US Dataset (mode=race)
Shape: (95309, 47) | Race: 0(Black)=19210, 1(White)=76099
  Final shape: (95309, 888)
  Sensitive groups: {0: 19210, 1: 76099}
✓ diabetes_race: (95309, 888)

Loading Diabetes 130-US Dataset (mode=combined)
Shape: (95309, 47) | Combined:
combined
0    11728
1     7482
2    39689
3    36410
Name: count, dtype: int64
0=Black-Female, 1=Black-Male, 2=White-Female, 3=White-Male
  Final shape: (95309, 888)
  Sensitive groups: {0: 11728, 1: 7482, 2: 39689, 3: 36410}
✓ diabetes_combined: (95309, 888)

Loading Dutch Census Dataset (mode=gender)
Loaded from URL. Shape: (60420, 12)
Gender: 0(F)=30273, 1(M)=30147
  Final shape: (60420, 10)
  Sensitive groups: {0: 30273, 1: 30147}
✓ dutch_gender: (60420, 10)

Loading MEPS Panel 20 (

## Checkpoint / Resume

In [13]:
# =============================================================================
# CHECKPOINT / RESUME — NEW
# -----------------------------------------------------------------------------
# k-selection: saved after every {dataset x method} (every dataset for Strategy 1).
# 30-run loop: saved after every single seed.
# If interrupted (crash / power loss / kernel died), re-run all cells from the top:
# finished parts are loaded from the checkpoint and skipped, and the loop continues
# from the seed where it stopped. Every seed calls np.random.seed(seed), so resumed
# results are identical to an uninterrupted run.
# To start over: set FRESH_START = True, run once (deletes the old checkpoint),
# then set it back to False.
# =============================================================================

import os, pickle, time

CKPT_PATH   = 'ckpt_S1B.pkl'   # checkpoint file for this notebook (same folder as the notebook)
FRESH_START = False

def _empty_ckpt():
    return {'ksel': {}, 'done': {}, 'partial': {}}

def save_ckpt(ck):
    # write to a temp file, then replace: a crash while saving cannot corrupt the old checkpoint
    tmp = CKPT_PATH + '.tmp'
    with open(tmp, 'wb') as f:
        pickle.dump(ck, f)
    os.replace(tmp, CKPT_PATH)

def load_ckpt():
    if FRESH_START and os.path.exists(CKPT_PATH):
        os.remove(CKPT_PATH)
        print(f"FRESH_START=True -> deleted old checkpoint {CKPT_PATH}")
    if os.path.exists(CKPT_PATH):
        with open(CKPT_PATH, 'rb') as f:
            ck = pickle.load(f)
        n_part = sum(len(p['seeds_done']) for p in ck['partial'].values())
        print(f"✓ Loaded checkpoint {CKPT_PATH}: k-selection done for {len(ck['ksel'])} item(s), "
              f"30-run done for {len(ck['done'])} dataset x method, "
              f"{len(ck['partial'])} partially done ({n_part} seed(s) in total)")
        return ck
    print(f"No checkpoint found ({CKPT_PATH}), starting from scratch")
    return _empty_ckpt()

ckpt = load_ckpt()

✓ Loaded checkpoint ckpt_S1B.pkl: k-selection done for 7 item(s), 30-run done for 17 dataset x method, 1 partially done (13 seed(s) in total)


## SECTION 5: K-SELECTION METHODS (unchanged from original)

In [15]:
# =============================================================================
# SECTION 5: K-SELECTION METHODS (unchanged from original)
# =============================================================================

def normalize_metric(values, inverse=False):
    values = np.array(values)
    mn, mx = np.min(values), np.max(values)
    if mx - mn == 0:
        return np.ones_like(values) * 0.5
    norm = (values - mn) / (mx - mn)
    return 1 - norm if inverse else norm


def approach1_optimal_k(X, k_range=(2, 11)):
    results = {'k_values': [], 'sse': [], 'silhouette': [],
                'calinski': [], 'dbi': [], 'silhouette_std': []}
    for k in range(k_range[0], k_range[1]):
        km = KMeans(n_clusters=k, random_state=42, n_init=10).fit(X)
        labels = km.labels_
        results['k_values'].append(k)
        results['sse'].append(km.inertia_)
        results['silhouette'].append(silhouette_score(X, labels))
        results['silhouette_std'].append(np.std(silhouette_samples(X, labels)))
        results['calinski'].append(calinski_harabasz_score(X, labels))
        results['dbi'].append(davies_bouldin_score(X, labels))
    combined = (normalize_metric(results['sse'], True) +
                normalize_metric(results['silhouette']) +
                normalize_metric(results['calinski']) +
                normalize_metric(results['dbi'], True) +
                normalize_metric(results['silhouette_std'], True)) / 5
    results['combined_scores'] = combined
    results['optimal_k'] = results['k_values'][np.argmax(combined)]
    return results


def elbow_method(X, k_range=(2, 11)):
    wcss = [KMeans(n_clusters=k, random_state=42, n_init=10).fit(X).inertia_
            for k in range(*k_range)]
    diff2 = np.diff(np.diff(wcss))
    idx = np.argmax(diff2) + 2
    idx = min(idx, len(range(*k_range)) - 1)
    return list(range(*k_range))[idx], wcss


def silhouette_method(X, k_range=(2, 11)):
    scores = [silhouette_score(X, KMeans(n_clusters=k, random_state=42, n_init=10).fit_predict(X))
              for k in range(*k_range)]
    return list(range(*k_range))[np.argmax(scores)], scores


def calinski_harabasz_method(X, k_range=(2, 11)):
    scores = [calinski_harabasz_score(X, KMeans(n_clusters=k, random_state=42, n_init=10).fit_predict(X))
              for k in range(*k_range)]
    return list(range(*k_range))[np.argmax(scores)], scores


def davies_bouldin_method(X, k_range=(2, 11)):
    scores = [davies_bouldin_score(X, KMeans(n_clusters=k, random_state=42, n_init=10).fit_predict(X))
              for k in range(*k_range)]
    return list(range(*k_range))[np.argmin(scores)], scores


def gap_statistic(X, k_range=(2, 11), n_refs=10):
    gaps = []
    for k in range(*k_range):
        wk = KMeans(n_clusters=k, random_state=42, n_init=10).fit(X).inertia_
        refs = [KMeans(n_clusters=k, random_state=None, n_init=10).fit(
                    np.random.uniform(X.min(0), X.max(0), X.shape)).inertia_
                for _ in range(n_refs)]
        gaps.append(np.log(np.mean(refs)) - np.log(wk))
    return list(range(*k_range))[np.argmax(gaps)], gaps


def stability_analysis(X, k_range=(2, 11), n_iterations=10, sample_ratio=0.8):
    n = int(len(X) * sample_ratio)
    stab = []
    for k in range(*k_range):
        aris = []
        for _ in range(n_iterations):
            i1, i2 = (np.random.choice(len(X), n, replace=False) for _ in range(2))
            common = np.intersect1d(i1, i2)
            if len(common) > k:
                l1 = KMeans(n_clusters=k, random_state=None, n_init=10).fit_predict(X[i1])
                l2 = KMeans(n_clusters=k, random_state=None, n_init=10).fit_predict(X[i2])
                c1 = np.array([np.where(i1 == x)[0][0] for x in common])
                c2 = np.array([np.where(i2 == x)[0][0] for x in common])
                aris.append(adjusted_rand_score(l1[c1], l2[c2]))
        stab.append(np.mean(aris) if aris else 0)
    return list(range(*k_range))[np.argmax(stab)], stab


def approach2_optimal_k(X, k_range=(2, 11)):
    r = {}
    print("  Elbow...", end='', flush=True)
    r['elbow_k'], r['elbow_scores'] = elbow_method(X, k_range)
    print(f" k={r['elbow_k']}")
    print("  Silhouette...", end='', flush=True)
    r['silhouette_k'], r['silhouette_scores'] = silhouette_method(X, k_range)
    print(f" k={r['silhouette_k']}")
    print("  Calinski-Harabasz...", end='', flush=True)
    r['calinski_k'], r['calinski_scores'] = calinski_harabasz_method(X, k_range)
    print(f" k={r['calinski_k']}")
    print("  Davies-Bouldin...", end='', flush=True)
    r['davies_k'], r['davies_scores'] = davies_bouldin_method(X, k_range)
    print(f" k={r['davies_k']}")
    print("  Gap Statistic...", end='', flush=True)
    r['gap_k'], r['gap_scores'] = gap_statistic(X, k_range)
    print(f" k={r['gap_k']}")
    print("  Stability...", end='', flush=True)
    r['stability_k'], r['stability_scores'] = stability_analysis(X, k_range)
    print(f" k={r['stability_k']}")
    votes = [r['elbow_k'], r['silhouette_k'], r['calinski_k'],
             r['davies_k'], r['gap_k'], r['stability_k']]
    cnt = Counter(votes)
    r['majority_vote_k'] = cnt.most_common(1)[0][0]
    r['k_distribution'] = dict(cnt)
    return r


# --- Run k-selection on all datasets ---
print("\n" + "="*80)
print("APPROACH 2: MAJORITY VOTE K SELECTION")
print("="*80)
approach2_results = {}
for name in dataset_names:
    print(f"\n{name.upper()}:")
    # NEW: k already selected -- load it from the checkpoint
    if name in ckpt['ksel']:
        approach2_results[name] = ckpt['ksel'][name]
        print(f"  	[restored from checkpoint] Majority Vote k = {approach2_results[name]['majority_vote_k']}")
        continue
    approach2_results[name] = approach2_optimal_k(datasets[name]['X'])
    ckpt['ksel'][name] = approach2_results[name]   # NEW
    save_ckpt(ckpt)                                # NEW
    print(f"  Majority Vote k = {approach2_results[name]['majority_vote_k']}")

# Summary table
summary_data = []
for name in dataset_names:
    row = {
        'Dataset': name,
        'Elbow': approach2_results[name]['elbow_k'],
        'Silhouette': approach2_results[name]['silhouette_k'],
        'Calinski': approach2_results[name]['calinski_k'],
        'Davies': approach2_results[name]['davies_k'],
        'Gap': approach2_results[name]['gap_k'],
        'Stability': approach2_results[name]['stability_k'],
        'Approach2_k': approach2_results[name]['majority_vote_k'],
    }
    summary_data.append(row)
summary_df = pd.DataFrame(summary_data)
print("\n" + "="*100)
print(summary_df.to_string(index=False))
summary_df.to_csv('optimal_k_summary_PartB.csv', index=False)
print("✓ Saved: optimal_k_summary_PartB.csv")


APPROACH 2: MAJORITY VOTE K SELECTION

DIABETES_GENDER:
  	[restored from checkpoint] Majority Vote k = 4

DIABETES_RACE:
  	[restored from checkpoint] Majority Vote k = 4

DIABETES_COMBINED:
  	[restored from checkpoint] Majority Vote k = 4

DUTCH_GENDER:
  	[restored from checkpoint] Majority Vote k = 10

MEPS_GENDER:
  	[restored from checkpoint] Majority Vote k = 5

MEPS_RACE:
  	[restored from checkpoint] Majority Vote k = 5

MEPS_COMBINED:
  	[restored from checkpoint] Majority Vote k = 5

          Dataset  Elbow  Silhouette  Calinski  Davies  Gap  Stability  Approach2_k
  diabetes_gender      4           2         3       2    9          4            4
    diabetes_race      4           2         3       7   10          6            4
diabetes_combined      4           2         3       7   10          4            4
     dutch_gender     10           2         9      10   10         10           10
      meps_gender      5           5         4       5   10          4        

## SECTION 6: CLUSTERING ALGORITHMS (create_fairlets, bfkm_clustering, PostProcessingNFP, PostProcessingGini generalized to multi-group -- Reviewer 3, Comment 1)

In [17]:
# =============================================================================
# SECTION 6: CLUSTERING ALGORITHMS (create_fairlets, bfkm_clustering, PostProcessingNFP, PostProcessingGini generalized to multi-group -- Reviewer 3, Comment 1)
# =============================================================================

def standard_kmeans(X, k, random_state=42):
    km = KMeans(n_clusters=k, random_state=random_state, n_init=10)
    return km.fit_predict(X), km.cluster_centers_


# ORIGINAL (Reviewer 3, Comment 1 -- kept for reference, not executed):
# def create_fairlets(X, sensitive_attr, p, q):
#     n = len(X)
#     g0, g1 = np.where(sensitive_attr == 0)[0], np.where(sensitive_attr == 1)[0]
#     fairlets, fairlet_labels = [], np.full(n, -1)
#     used0, used1, fid = set(), set(), 0
#     while len(used0)+p <= len(g0) and len(used1)+q <= len(g1):
#         s0 = np.random.choice([i for i in g0 if i not in used0], p, replace=False)
#         s1 = np.random.choice([i for i in g1 if i not in used1], q, replace=False)
#         members = np.concatenate([s0, s1])
#         fairlets.append(members)
#         fairlet_labels[members] = fid
#         used0.update(s0); used1.update(s1); fid += 1
#     rem = np.where(fairlet_labels == -1)[0]
#     if len(rem):
#         fairlets.append(rem); fairlet_labels[rem] = fid
#     return fairlets, fairlet_labels

# Fairlet decomposition, generalised to multiple groups (our own extension).
# Chierichetti et al. (NeurIPS 2017) define fairlets for two groups only; no
# published multi-group fairlet algorithm exists. Here every fairlet contains
# points from every group, with each group's share set by rounding its count
# relative to the smallest group (in the spirit of Bera et al., NeurIPS 2019).
# The same rule is used for two or more groups.
# Limitation (disclosed in the manuscript): the original (b, r)-fairlet uses
# the exact group ratio in lowest terms; we use a rounded approximation to keep
# fairlets small (e.g. a 60:40 split becomes 2:1 rather than 3:2), because an
# exact reduction of real counts (e.g. gcd(21790, 10771) = 1) would require
# impractically large fairlets.
# `p`, `q` are unused and kept only for backward compatibility.


def create_fairlets(X, sensitive_attr, p=1, q=1):
    n = len(X)
    S = np.asarray(sensitive_attr)
    groups = np.unique(S)

    group_idxs = {g: np.where(S == g)[0] for g in groups}
    counts = {g: len(group_idxs[g]) for g in groups}
    min_count = max(1, min(counts.values()))
    unit_sizes = {g: max(1, round(counts[g] / min_count)) for g in groups}

    fairlets, fairlet_labels = [], np.full(n, -1)
    unused = {g: np.ones(counts[g], dtype=bool) for g in groups} 
    fid = 0
    while True:
        members, ok = [], True
        for g in groups:
            avail = group_idxs[g][unused[g]]   
            if len(avail) < unit_sizes[g]:
                ok = False
                break
            sel = np.random.choice(avail, unit_sizes[g], replace=False)
            members.append(sel)
            unused[g][np.searchsorted(group_idxs[g], sel)] = False
        if not ok:
            break
        members = np.concatenate(members)
        fairlets.append(members)
        fairlet_labels[members] = fid
        fid += 1
    rem = np.where(fairlet_labels == -1)[0]
    if len(rem):
        fairlets.append(rem)
        fairlet_labels[rem] = fid
    return fairlets, fairlet_labels


def fairlet_clustering(X, sensitive_attr, k, p=1, q=1):
    fairlets, _ = create_fairlets(X, sensitive_attr, p, q)
    centers_f = np.array([np.mean(X[f], axis=0) for f in fairlets])
    fc_labels = KMeans(n_clusters=min(k, len(centers_f)), random_state=42, n_init=10).fit_predict(centers_f) \
        if len(centers_f) >= k else np.arange(len(centers_f))
    labels = np.zeros(len(X), dtype=int)
    for i, cl in enumerate(fc_labels):
        labels[fairlets[i]] = cl
    centers = np.array([np.mean(X[labels == i], axis=0) if (labels == i).any()
                         else np.zeros(X.shape[1]) for i in range(k)])
    return labels, centers


# ORIGINAL (Reviewer 3, Comment 1 & 7 -- kept for reference, not executed):
# def bfkm_clustering(X, sensitive_attr, k, max_iter=100):
#     centers = X[np.random.choice(len(X), k, replace=False)].copy()
#     for _ in range(max_iter):
#         old = centers.copy()
#         dists = cdist(X, centers)
#         labels = np.argmin(dists, axis=1)
#         t0, t1 = (sensitive_attr == 0).sum(), (sensitive_attr == 1).sum()
#         for c in range(k):
#             mask = labels == c
#             s = sensitive_attr[mask]
#             n0, n1 = (s == 0).sum(), (s == 1).sum()
#             if t0 > 0 and t1 > 0:
#                 ratio_exp = t1 / t0
#                 ratio_act = n1 / max(n0, 1)
#                 if abs(ratio_act - ratio_exp) > 0.3:
#                     idxs = np.where(mask)[0]
#                     grp = 1 if ratio_act > ratio_exp else 0
#                     g_idxs = idxs[sensitive_attr[idxs] == grp]
#                     if len(g_idxs):
#                         far = g_idxs[np.argmax(dists[g_idxs, c])]
#                         sc = np.argsort(dists[far])
#                         labels[far] = sc[1] if sc[1] != c else sc[2]
#         for i in range(k):
#             pts = X[labels == i]
#             if len(pts): centers[i] = pts.mean(0)
#         if np.allclose(centers, old, rtol=1e-4): break
#     return labels, centers

# FIXED (Reviewer 3, Comment 1 & 7): generalized from binary (groups 0/1
# hardcoded) to any number of groups, matching the demographic-parity
# constraint of the original method (Pan, Zhong & Qian, 2023):
# P(cluster=j | group=l) = P(cluster=j) for every cluster j and group l --
# equivalently, each group's within-cluster proportion should match its
# global proportion. The solver itself remains the same greedy heuristic
# (find the cluster's most over-represented group relative to its global
# share, i.e. the largest POSITIVE deviation, and move its farthest point
# out; under-represented groups are never moved out), not the paper's exact
# Lagrangian-relaxed alternating optimization, and without its cluster-size
# balance term -- generalized in what it checks (all groups vs. a global
# reference), not in how it solves the optimization.

def bfkm_clustering(X, sensitive_attr, k, max_iter=100):
    centers = X[np.random.choice(len(X), k, replace=False)].copy()
    S = np.asarray(sensitive_attr)
    groups = np.unique(S)
    global_props = {g: (S == g).sum() / len(S) for g in groups}
    for _ in range(max_iter):
        old = centers.copy()
        dists = cdist(X, centers)
        labels = np.argmin(dists, axis=1)
        for c in range(k):
            mask = labels == c
            n_c = mask.sum()
            if n_c == 0:
                continue
            s = S[mask]
            worst_g, worst_dev = None, 0.0
            for g in groups:
                prop_c = (s == g).sum() / n_c
                dev = prop_c - global_props[g]
                if dev > worst_dev:
                    worst_g, worst_dev = g, dev
            if worst_g is not None and worst_dev > 0.3:
                idxs = np.where(mask)[0]
                g_idxs = idxs[S[idxs] == worst_g]
                if len(g_idxs):
                    far = g_idxs[np.argmax(dists[g_idxs, c])]
                    sc = np.argsort(dists[far])
                    labels[far] = sc[1] if sc[1] != c else sc[2]
        for i in range(k):
            pts = X[labels == i]
            if len(pts): centers[i] = pts.mean(0)
        if np.allclose(centers, old, rtol=1e-4): break
    return labels, centers


# ORIGINAL (runtime bottleneck -- kept for reference, not executed):
# class OptimizedFairCentroid:
#     def __init__(self, n_clusters=6, max_iter=50, min_iter=10,
#                  tol=0.005, random_state=0, candidate_fraction=0.05):
#         self.n_clusters = n_clusters
#         self.max_iter = max_iter; self.min_iter = min_iter
#         self.tol = tol; self.random_state = random_state
#         self.candidate_fraction = candidate_fraction
#         self.labels_ = self.cluster_centers_ = self.inertia_ = None; self.n_iter_ = 0
#
#     def _compute_cluster_fairness_fast(self, X_c, S_c, centroid):
#         if len(X_c) == 0: return 0.0
#         uq = np.unique(S_c)
#         if len(uq) < 2: return 0.0
#         d2 = np.sum((X_c - centroid)**2, axis=1)
#         gm = [d2[S_c == g].mean() for g in uq if (S_c == g).sum() > 0]
#         return max(gm) - min(gm) if len(gm) >= 2 else 0.0
#
#     def _precompute_cluster_stats(self, X, labels, S, centers):
#         stats = {}
#         for c in range(self.n_clusters):
#             mask = labels == c; n_c = mask.sum()
#             if n_c == 0:
#                 stats[c] = {'n': 0, 'centroid': centers[c], 'fairness': 0.0, 'X': np.array([]), 'S': np.array([])}
#             else:
#                 Xc, Sc = X[mask], S[mask]
#                 stats[c] = {'n': n_c, 'centroid': centers[c],
#                             'fairness': self._compute_cluster_fairness_fast(Xc, Sc, centers[c]),
#                             'X': Xc, 'S': Sc}
#         return stats
#
#     def _find_candidates(self, X, labels, centers, n_candidates):
#         n = len(X)
#         nc = max(1, int(n * self.candidate_fraction))
#         dists = np.array([np.linalg.norm(X[i] - centers[labels[i]]) for i in range(n)])
#         return np.argsort(-dists)[:nc]
#
#     def fit(self, X, sensitive_attr):
#         S = np.array(sensitive_attr if not hasattr(sensitive_attr, 'values') else sensitive_attr.values)
#         km = KMeans(n_clusters=self.n_clusters, random_state=self.random_state, n_init=10)
#         self.labels_ = km.fit_predict(X)
#         centers = km.cluster_centers_.copy()
#         for it in range(self.max_iter):
#             stats = self._precompute_cluster_stats(X, self.labels_, S, centers)
#             cands = self._find_candidates(X, self.labels_, centers, int(len(X)*self.candidate_fraction))
#             improved = False
#             for idx in cands:
#                 cur = self.labels_[idx]
#                 cur_fair = stats[cur]['fairness']
#                 best_c, best_delta = cur, 0
#                 for c2 in range(self.n_clusters):
#                     if c2 == cur: continue
#                     new_labels = self.labels_.copy()
#                     new_labels[idx] = c2
#                     new_centers = centers.copy()
#                     for ci in [cur, c2]:
#                         pts = X[new_labels == ci]
#                         if len(pts): new_centers[ci] = pts.mean(0)
#                     new_stats = self._precompute_cluster_stats(X, new_labels, S, new_centers)
#                     delta = (stats[cur]['fairness'] + stats[c2]['fairness']) - \
#                             (new_stats[cur]['fairness'] + new_stats[c2]['fairness'])
#                     if delta > best_delta:
#                         best_delta, best_c = delta, c2
#                 if best_c != cur:
#                     self.labels_[idx] = best_c
#                     for ci in [cur, best_c]:
#                         pts = X[self.labels_ == ci]
#                         if len(pts): centers[ci] = pts.mean(0)
#                     stats = self._precompute_cluster_stats(X, self.labels_, S, centers)
#                     improved = True
#             if it >= self.min_iter and not improved: break
#         self.cluster_centers_ = centers
#         self.n_iter_ = it + 1
#         return self

# CCF (Cluster-level Centroid Fairness), optimised implementation.
# Same algorithm as the original version (kept above, commented out): standard
# K-means (k-means++, n_init=10), then greedy local-search reassignment of the
# 5% of points farthest from their centroid, each moved to the cluster that most
# reduces the per-cluster disparity in mean squared distance-to-centroid between
# sensitive groups; stops when there is no gain after min_iter, or at max_iter.
# Only the evaluation of a candidate move is faster: per-(cluster, group) counts,
# coordinate sums and squared-norm sums give the post-move fairness in closed form,
#     mean ||x - mu||^2 = (sum ||x||^2 - 2 mu . sum x + n ||mu||^2) / n,
# instead of recomputing all clusters over the full data (O(groups*d) vs O(n*d*k)).
# Statistics are recomputed from the data at the start of every iteration.
# Verified to give labels identical to the original implementation.

class OptimizedFairCentroid:
    def __init__(self, n_clusters=6, max_iter=50, min_iter=10,
                 tol=0.005, random_state=0, candidate_fraction=0.05):
        self.n_clusters = n_clusters
        self.max_iter = max_iter; self.min_iter = min_iter
        self.tol = tol; self.random_state = random_state
        self.candidate_fraction = candidate_fraction
        self.labels_ = self.cluster_centers_ = self.inertia_ = None; self.n_iter_ = 0

    # unchanged from the original implementation
    def _compute_cluster_fairness_fast(self, X_c, S_c, centroid):
        if len(X_c) == 0: return 0.0
        uq = np.unique(S_c)
        if len(uq) < 2: return 0.0
        d2 = np.sum((X_c - centroid)**2, axis=1)
        gm = [d2[S_c == g].mean() for g in uq if (S_c == g).sum() > 0]
        return max(gm) - min(gm) if len(gm) >= 2 else 0.0

    # unchanged from the original implementation
    def _find_candidates(self, X, labels, centers, n_candidates):
        n = len(X)
        nc = max(1, int(n * self.candidate_fraction))
        dists = np.array([np.linalg.norm(X[i] - centers[labels[i]]) for i in range(n)])
        return np.argsort(-dists)[:nc]

    @staticmethod
    def _fair_from_stats(cnt_c, sx_c, sq_c, mu):
        # same quantity as _compute_cluster_fairness_fast, from sufficient statistics
        present = cnt_c > 0
        if present.sum() < 2:
            return 0.0
        n = cnt_c[present]
        gm = (sq_c[present] - 2.0 * (sx_c[present] @ mu) + n * (mu @ mu)) / n
        return float(gm.max() - gm.min())

    def fit(self, X, sensitive_attr):
        S = np.array(sensitive_attr if not hasattr(sensitive_attr, 'values') else sensitive_attr.values)
        km = KMeans(n_clusters=self.n_clusters, random_state=self.random_state, n_init=10)
        self.labels_ = km.fit_predict(X)
        centers = km.cluster_centers_.copy()
        labels = self.labels_
        k = self.n_clusters
        groups = np.unique(S)
        g_idx = np.searchsorted(groups, S)
        G = len(groups)
        sqn = np.einsum('ij,ij->i', X, X)
        touched = np.zeros(k, dtype=bool)

        for it in range(self.max_iter):
            # --- iteration start: recompute directly from the data (as original) ---
            for c in np.where(touched)[0]:
                pts = X[labels == c]
                if len(pts): centers[c] = pts.mean(0)
            touched[:] = False
            cnt = np.zeros((k, G)); sx = np.zeros((k, G, X.shape[1])); sq = np.zeros((k, G))
            fair = np.zeros(k)
            for c in range(k):
                mask = labels == c
                if mask.any():
                    fair[c] = self._compute_cluster_fairness_fast(X[mask], S[mask], centers[c])
                    for g in range(G):
                        m = mask & (g_idx == g)
                        cnt[c, g] = m.sum(); sx[c, g] = X[m].sum(0); sq[c, g] = sqn[m].sum()

            cands = self._find_candidates(X, labels, centers, int(len(X)*self.candidate_fraction))
            improved = False
            for idx in cands:
                cur, g, x, q = labels[idx], g_idx[idx], X[idx], sqn[idx]
                # cluster `cur` after removing the point
                cnt_a = cnt[cur].copy(); cnt_a[g] -= 1
                sx_a = sx[cur].copy();   sx_a[g] -= x
                sq_a = sq[cur].copy();   sq_a[g] -= q
                n_a = cnt_a.sum()
                mu_a = sx_a.sum(0) / n_a if n_a > 0 else centers[cur]
                f_a = self._fair_from_stats(cnt_a, sx_a, sq_a, mu_a) if n_a > 0 else 0.0
                best_c, best_delta = cur, 0
                for c2 in range(k):
                    if c2 == cur: continue
                    # cluster `c2` after adding the point
                    cnt_b = cnt[c2].copy(); cnt_b[g] += 1
                    sx_b = sx[c2].copy();   sx_b[g] += x
                    sq_b = sq[c2].copy();   sq_b[g] += q
                    mu_b = sx_b.sum(0) / cnt_b.sum()
                    delta = (fair[cur] + fair[c2]) - \
                            (f_a + self._fair_from_stats(cnt_b, sx_b, sq_b, mu_b))
                    if delta > best_delta:
                        best_delta, best_c = delta, c2
                if best_c != cur:
                    labels[idx] = best_c
                    cnt[cur, g] -= 1; sx[cur, g] -= x; sq[cur, g] -= q
                    cnt[best_c, g] += 1; sx[best_c, g] += x; sq[best_c, g] += q
                    for ci in (cur, best_c):
                        n_ci = cnt[ci].sum()
                        if n_ci > 0:
                            centers[ci] = sx[ci].sum(0) / n_ci
                            touched[ci] = True
                        fair[ci] = self._fair_from_stats(cnt[ci], sx[ci], sq[ci], centers[ci]) if n_ci > 0 else 0.0
                    improved = True
            if it >= self.min_iter and not improved: break

        # final centroids recomputed directly from the data, as in the original
        for c in np.where(touched)[0]:
            pts = X[labels == c]
            if len(pts): centers[c] = pts.mean(0)
        self.cluster_centers_ = centers
        self.n_iter_ = it + 1
        return self


def fair_centroid_clustering(X, sensitive_attr, k, max_iter=20, min_iter=5, random_state=0):
    model = OptimizedFairCentroid(n_clusters=k, max_iter=max_iter,
                                   min_iter=min_iter, random_state=random_state)
    model.fit(X, sensitive_attr)
    return model.labels_, model.cluster_centers_

# ORIGINAL (Reviewer 3, Comment 1 -- kept for reference, not executed):
# class PostProcessingNFP:
#     def __init__(self, n_clusters=6, balance_tolerance=0.05, max_iter=30, random_state=0):
#         self.n_clusters = n_clusters; self.balance_tolerance = balance_tolerance
#         self.max_iter = max_iter; self.random_state = random_state
#         self.labels_ = self.cluster_centers_ = None
#
#     def fit(self, X, sensitive_attr):
#         S = np.array(sensitive_attr)
#         km = KMeans(n_clusters=self.n_clusters, random_state=self.random_state, n_init=10)
#         self.labels_ = km.fit_predict(X)
#         centers = km.cluster_centers_.copy()
#         t0, t1 = (S == 0).sum(), (S == 1).sum()
#         global_ratio = t1 / max(t0, 1)
#         for _ in range(self.max_iter):
#             changed = False
#             ratios = []
#             for c in range(self.n_clusters):
#                 mask = self.labels_ == c
#                 s = S[mask]
#                 ratios.append((s == 1).sum() / max((s == 0).sum(), 1))
#             max_c, min_c = np.argmax(ratios), np.argmin(ratios)
#             if abs(ratios[max_c] - global_ratio) <= self.balance_tolerance: break
#             mask_max = np.where((self.labels_ == max_c) & (S == 1))[0]
#             if len(mask_max) == 0: break
#             dists_to_min = np.linalg.norm(X[mask_max] - centers[min_c], axis=1)
#             nearest = mask_max[np.argmin(dists_to_min)]
#             self.labels_[nearest] = min_c
#             for ci in [max_c, min_c]:
#                 pts = X[self.labels_ == ci]
#                 if len(pts): centers[ci] = pts.mean(0)
#             changed = True
#             if not changed: break
#         self.cluster_centers_ = centers
#         return self

# FIXED (Reviewer 3, Comment 1): generalized from binary (groups 0/1
# hardcoded) to any number of groups. Previously this class only ever
# compared group-1-vs-group-0 ratios, so for _combined (4-group)
# configurations groups 2 and 3 were never checked during reassignment.
# Now finds the (cluster, group) pair with the worst positive deviation of
# within-cluster proportion from global proportion (the same
# demographic-parity criterion used for the generalized BFKM above), and
# reassigns that group's nearest-to-target-centroid point to whichever
# other cluster most under-represents that same group -- preserving the
# original "nearest foreign point" mechanism, generalized to multiple
# groups instead of a single binary comparison.

class PostProcessingNFP:
    def __init__(self, n_clusters=6, balance_tolerance=0.05, max_iter=30, random_state=0):
        self.n_clusters = n_clusters; self.balance_tolerance = balance_tolerance
        self.max_iter = max_iter; self.random_state = random_state
        self.labels_ = self.cluster_centers_ = None

    def fit(self, X, sensitive_attr):
        S = np.array(sensitive_attr)
        groups = np.unique(S)
        global_props = {g: (S == g).sum() / len(S) for g in groups}
        km = KMeans(n_clusters=self.n_clusters, random_state=self.random_state, n_init=10)
        self.labels_ = km.fit_predict(X)
        centers = km.cluster_centers_.copy()
        for _ in range(self.max_iter):
            worst_c, worst_g, worst_dev = None, None, 0.0
            for c in range(self.n_clusters):
                mask = self.labels_ == c
                n_c = mask.sum()
                if n_c == 0:
                    continue
                s = S[mask]
                for g in groups:
                    dev = (s == g).sum() / n_c - global_props[g]
                    if dev > worst_dev:
                        worst_c, worst_g, worst_dev = c, g, dev
            if worst_c is None or worst_dev <= self.balance_tolerance:
                break
            under_c, under_dev = None, 0.0
            for c in range(self.n_clusters):
                if c == worst_c:
                    continue
                mask = self.labels_ == c
                n_c = mask.sum()
                if n_c == 0:
                    continue
                dev = global_props[worst_g] - (S[mask] == worst_g).sum() / n_c
                if dev > under_dev:
                    under_c, under_dev = c, dev
            if under_c is None:
                break
            cand = np.where((self.labels_ == worst_c) & (S == worst_g))[0]
            if len(cand) == 0:
                break
            dists_to_target = np.linalg.norm(X[cand] - centers[under_c], axis=1)
            nearest = cand[np.argmin(dists_to_target)]
            self.labels_[nearest] = under_c
            for ci in [worst_c, under_c]:
                pts = X[self.labels_ == ci]
                if len(pts): centers[ci] = pts.mean(0)
        self.cluster_centers_ = centers
        return self


# ORIGINAL (Reviewer 3, Comment 1 -- kept for reference, not executed):
# class PostProcessingGini:
#     def __init__(self, n_clusters=6, k_neighbors=10, balance_tolerance=0.05, max_iter=30, random_state=0):
#         self.n_clusters = n_clusters; self.k_neighbors = k_neighbors
#         self.balance_tolerance = balance_tolerance; self.max_iter = max_iter
#         self.random_state = random_state; self.labels_ = self.cluster_centers_ = None
#
#     def _compute_gini_coefficient(self, group_counts):
#         if sum(group_counts.values()) == 0: return 0
#         total = sum(group_counts.values())
#         probs = [v/total for v in group_counts.values()]
#         return 1 - sum(p**2 for p in probs)
#
#     def _compute_overall_gini(self, labels, S, k):
#         ginis = []
#         for c in range(k):
#             mask = labels == c
#             if mask.sum() == 0: continue
#             s = S[mask]
#             counts = {g: (s == g).sum() for g in np.unique(S)}
#             ginis.append(self._compute_gini_coefficient(counts))
#         return np.mean(ginis) if ginis else 0
#
#     def fit(self, X, sensitive_attr):
#         S = np.array(sensitive_attr)
#         km = KMeans(n_clusters=self.n_clusters, random_state=self.random_state, n_init=10)
#         self.labels_ = km.fit_predict(X)
#         centers = km.cluster_centers_.copy()
#         t0, t1 = (S == 0).sum(), (S == 1).sum()
#         global_ratio = t1 / max(t0, 1)
#         for _ in range(self.max_iter):
#             ratios = [(S[(self.labels_ == c) & True] == 1).sum() /
#                       max((S[self.labels_ == c] == 0).sum(), 1) for c in range(self.n_clusters)]
#             max_c, min_c = np.argmax(ratios), np.argmin(ratios)
#             if abs(ratios[max_c] - global_ratio) <= self.balance_tolerance: break
#             # Gini-guided selection
#             cands = np.where((self.labels_ == max_c) & (S == 1))[0]
#             if len(cands) == 0: break
#             ginis = []
#             for idx in cands:
#                 tmp = self.labels_.copy(); tmp[idx] = min_c
#                 ginis.append(self._compute_overall_gini(tmp, S, self.n_clusters))
#             best = cands[np.argmax(ginis)]
#             self.labels_[best] = min_c
#             for ci in [max_c, min_c]:
#                 pts = X[self.labels_ == ci]
#                 if len(pts): centers[ci] = pts.mean(0)
#         self.cluster_centers_ = centers
#         return self

# FIXED (Reviewer 3, Comment 1): Gini now chooses the move (source cluster,
# group, target cluster) from the cluster x group counts; distance chooses
# the point (nearest to the target centroid, same rule as PostProcessingNFP).
# Unlike PostProcessingNFP, which always moves the worst over-represented
# group to the cluster that most under-represents it, the move here is
# whichever one most increases the mean Gini impurity.
# Generalized from binary to any number of groups via np.unique(S): the
# original loop compared only group 1 vs group 0, so for _combined (4-group)
# configurations groups 2 and 3 were never considered.
# Stops at max deviation <= 0.05, no Gini gain, or max_iter.

class PostProcessingGini:
    def __init__(self, n_clusters=6, k_neighbors=10, balance_tolerance=0.05, max_iter=30, random_state=0):
        self.n_clusters = n_clusters; self.k_neighbors = k_neighbors
        self.balance_tolerance = balance_tolerance; self.max_iter = max_iter
        self.random_state = random_state; self.labels_ = self.cluster_centers_ = None
 
    @staticmethod
    def _mean_gini(counts):
        # counts: (n_clusters x n_groups); mean Gini impurity over non-empty clusters
        sizes = counts.sum(1)
        nonempty = sizes > 0
        if not nonempty.any(): return 0.0
        p = counts[nonempty] / sizes[nonempty, None]
        return float(np.mean(1 - (p ** 2).sum(1)))
 
    def fit(self, X, sensitive_attr):
        S = np.array(sensitive_attr)
        groups = np.unique(S)
        g_idx = np.searchsorted(groups, S)
        global_props = np.bincount(g_idx, minlength=len(groups)) / len(S)
        km = KMeans(n_clusters=self.n_clusters, random_state=self.random_state, n_init=10)
        self.labels_ = km.fit_predict(X)
        centers = km.cluster_centers_.copy()
        for _ in range(self.max_iter):
            counts = np.zeros((self.n_clusters, len(groups)))
            np.add.at(counts, (self.labels_, g_idx), 1)
            sizes = counts.sum(1, keepdims=True)
            dev = np.where(sizes > 0, counts / np.maximum(sizes, 1) - global_props, 0.0)
            if dev.max() <= self.balance_tolerance:
                break
            # Gini-guided choice of the move
            base = self._mean_gini(counts)
            best_gain, best_move = 1e-12, None
            for c in range(self.n_clusters):
                if sizes[c, 0] <= 1:
                    continue                      # never empty a cluster
                for g in range(len(groups)):
                    if counts[c, g] == 0:
                        continue
                    for c2 in range(self.n_clusters):
                        if c2 == c:
                            continue
                        counts[c, g] -= 1; counts[c2, g] += 1
                        gain = self._mean_gini(counts) - base
                        counts[c, g] += 1; counts[c2, g] -= 1
                        if gain > best_gain:
                            best_gain, best_move = gain, (c, g, c2)
            if best_move is None:
                break
            c, g, c2 = best_move
            # distance-guided choice of the point
            cand = np.where((self.labels_ == c) & (g_idx == g))[0]
            dists_to_target = np.linalg.norm(X[cand] - centers[c2], axis=1)
            nearest = cand[np.argmin(dists_to_target)]
            self.labels_[nearest] = c2
            for ci in [c, c2]:
                pts = X[self.labels_ == ci]
                if len(pts): centers[ci] = pts.mean(0)
        self.cluster_centers_ = centers
        return self

def postprocessing_clustering_nfp(X, sensitive_attr, k, max_iter=30, random_state=0):
    m = PostProcessingNFP(n_clusters=k, max_iter=max_iter, random_state=random_state)
    m.fit(X, sensitive_attr)
    return m.labels_, m.cluster_centers_


def postprocessing_clustering_gini(X, sensitive_attr, k, max_iter=30, random_state=0):
    m = PostProcessingGini(n_clusters=k, max_iter=max_iter, random_state=random_state)
    m.fit(X, sensitive_attr)
    return m.labels_, m.cluster_centers_
    

def rawlsian_kmeans(X, sensitive_attr, k, n_runs=30, delta=None, random_state=0):
    Xd = X.toarray() if hasattr(X, 'toarray') else X
    S = sensitive_attr.values if hasattr(sensitive_attr, 'values') else np.array(sensitive_attr)
    if delta is None: delta = np.sqrt(Xd.shape[1])
    best_score, best_labels, best_centers = -np.inf, None, None
    for seed in range(n_runs):
        km = KMeans(n_clusters=k, random_state=random_state+seed, n_init=1)
        labels = km.fit_predict(Xd); centers = km.cluster_centers_
        utils = np.maximum(0, 1 - np.linalg.norm(Xd - centers[labels], axis=1) / delta)
        gu = [utils[S == g].mean() for g in np.unique(S) if (S == g).sum() > 0]
        score = min(gu) if gu else 0
        if score > best_score:
            best_score, best_labels, best_centers = score, labels.copy(), centers.copy()
    return best_labels, best_centers

## SECTION 7: METRICS (Balance, SPD, Disparate_Impact, Entropy fixed to match Eq. 5-8 and generalized to multi-group -- Reviewer 3, Comment 1 & 2)

In [19]:
# =============================================================================
# SECTION 7: METRICS (Balance, SPD, Disparate_Impact, Entropy fixed to match Eq. 5-8 and generalized to multi-group -- Reviewer 3, Comment 1 & 2)
# =============================================================================

def calculate_quality_metrics(X, labels):
    k = len(np.unique(labels))
    centers = np.array([X[labels==i].mean(0) if (labels==i).any() else np.zeros(X.shape[1])
                         for i in range(k)])
    inertia = sum(np.sum((X[labels==i]-centers[i])**2) for i in range(k))
    sil = silhouette_score(X, labels) if 1 < k < len(X) else 0
    ch  = calinski_harabasz_score(X, labels) if 1 < k < len(X) else 0
    db  = davies_bouldin_score(X, labels) if 1 < k < len(X) else 0
    overall_mean = X.mean(0)
    wcss = sum(np.sum((X[labels==i]-centers[i])**2) for i in range(k))
    bcss = sum(len(X[labels==i])*np.sum((centers[i]-overall_mean)**2) for i in range(k))
    tss = np.sum((X-overall_mean)**2)
    vr = bcss/wcss if wcss > 0 else 0
    try:
        icd = np.mean(pdist(centers)) if k > 1 else 0
    except: icd = 0
    intra_d = sep = dunn = np.nan
    return {'Inertia': inertia, 'Silhouette': sil, 'Calinski_Harabasz': ch,
            'Davies_Bouldin': db, 'Dunn_Index': dunn, 'WCSS': wcss, 'BCSS': bcss,
            'TSS': tss, 'Variance_Ratio': vr, 'Inter_Cluster_Distance': icd,
            'Intra_Cluster_Distance': intra_d, 'Separation_Index': sep}


# ORIGINAL (Reviewer 3, Comment 1 & 2 -- kept for reference, not executed):
# def calculate_balance(labels, S):
#     balances = []
#     for c in np.unique(labels):
#         sc = S[labels==c]
#         n0, n1 = (sc==0).sum(), (sc==1).sum()
#         if n0 > 0 and n1 > 0:
#             balances.append(min(n0,n1)/max(n0,n1))
#     return np.mean(balances) if balances else 0.0
#
#
# def calculate_spd(labels, S):
#     t0, t1 = (S==0).sum(), (S==1).sum()
#     vals = []
#     for c in np.unique(labels):
#         mask = labels==c
#         p0 = (S[mask]==0).sum()/t0 if t0 > 0 else 0
#         p1 = (S[mask]==1).sum()/t1 if t1 > 0 else 0
#         vals.append(abs(p0-p1))
#     return np.mean(vals)
#
#
# def calculate_disparate_impact(labels, S):
#     t0, t1 = (S==0).sum(), (S==1).sum()
#     vals = []
#     for c in np.unique(labels):
#         mask = labels==c
#         p0 = (S[mask]==0).sum()/t0 if t0 > 0 else 0
#         p1 = (S[mask]==1).sum()/t1 if t1 > 0 else 0
#         if p0 > 0 and p1 > 0:
#             vals.append(min(p0,p1)/max(p0,p1))
#     return np.mean(vals) if vals else 0.0
#
#
# def calculate_entropy_metric(labels, S):
#     entropies = []
#     for c in np.unique(labels):
#         sc = S[labels==c]
#         n0, n1 = (sc==0).sum(), (sc==1).sum()
#         if n0 > 0 and n1 > 0:
#             p = np.array([n0, n1]) / len(sc)
#             entropies.append(entropy(p, base=2))
#     return np.mean(entropies) if entropies else 0.0

# FIXED (Reviewer 3, Comment 1): generalized from binary (groups 0/1
# hardcoded) to any number of groups via np.unique(S), so _combined
# (4-group) configurations are no longer silently reduced to a 2-group
# comparison.
# FIXED (Reviewer 3, Comment 2): Balance changed from a mean across
# clusters to the worst-cluster minimum, matching Eq. (7).

def calculate_balance(labels, S):
    groups = np.unique(S)
    balances = []
    for c in np.unique(labels):
        sc = S[labels == c]
        props = np.array([(sc == g).sum() for g in groups]) / max(len(sc), 1)
        balances.append(props.min() / props.max() if props.max() > 0 else 0.0)
    return min(balances) if balances else 0.0


# FIXED (Reviewer 3, Comment 1 & 2): generalized to any number of groups;
# changed from a mean of |p0-p1| between two hardcoded groups to the
# maximum absolute deviation of within-cluster group proportion from the
# group's global proportion, matching Eq. (5): SPD = max_{k,a} |P(a|Ck) - P(a)|
def calculate_spd(labels, S):
    groups = np.unique(S)
    global_props = {g: (S == g).sum() / len(S) for g in groups}
    vals = []
    for c in np.unique(labels):
        mask = labels == c
        n_c = mask.sum()
        if n_c == 0:
            continue
        for g in groups:
            p_a_given_ck = (S[mask] == g).sum() / n_c
            vals.append(abs(p_a_given_ck - global_props[g]))
    return max(vals) if vals else 0.0


# FIXED (Reviewer 3, Comment 1 & 2): generalized to any number of groups;
# now returns the RAW Disparate Impact value (ideal = 1), matching Eq. (6):
# DI = min_{k,a} P(a|Ck) / P(a). Previously this function silently mixed a
# binary min/max ratio with an aggregation elsewhere that treated smaller
# raw values as "better" -- direction is now handled explicitly via
# DI_dist = |DI - 1| wherever this value is used for ranking (see run_single
# below), rather than baked into this function.
def calculate_disparate_impact(labels, S):
    groups = np.unique(S)
    global_props = {g: (S == g).sum() / len(S) for g in groups}
    vals = []
    for c in np.unique(labels):
        mask = labels == c
        n_c = mask.sum()
        if n_c == 0:
            continue
        for g in groups:
            p_a = global_props[g]
            if p_a > 0:
                p_a_given_ck = (S[mask] == g).sum() / n_c
                vals.append(p_a_given_ck / p_a)
    return min(vals) if vals else 0.0


# FIXED (Reviewer 3, Comment 2): generalized to any number of groups (was
# hardcoded to n0, n1); pure clusters are now INCLUDED with entropy = 0
# rather than excluded, since a pure cluster is a real (maximally unequal)
# outcome that should count toward the average, not be skipped.
def calculate_entropy_metric(labels, S):
    groups = np.unique(S)
    entropies = []
    for c in np.unique(labels):
        sc = S[labels == c]
        if len(sc) == 0:
            continue
        counts = np.array([(sc == g).sum() for g in groups])
        p = counts / counts.sum()
        p = p[p > 0]  # drop zero-probability terms so entropy() doesn't warn; result is unaffected (0*log(0) := 0)
        entropies.append(entropy(p, base=2))  # = 0 for a pure cluster, now included rather than excluded
    return np.mean(entropies) if entropies else 0.0

## SECTION 8: MAIN EXPERIMENT LOOP

## 30-Run Experiment Loop

In [22]:
import time

SEEDS        = list(range(30))

# ORIGINAL (Reviewer 3, Comment 4 & 6 -- kept for reference, not executed):
# UTIL_COLS = ['Inertia','Silhouette','Calinski_Harabasz','Davies_Bouldin','BCSS','Variance_Ratio']
# FAIR_COLS = ['Balance','SPD','Disparate_Impact','Entropy']
#
# def run_single(X, sensitive, k, method_name, seed):
#     np.random.seed(seed)
#     if method_name == 'kmeans':
#         labels, centers = standard_kmeans(X, k, random_state=seed)
#     elif method_name == 'fairlet':
#         labels, centers = fairlet_clustering(X, sensitive, k)
#     elif method_name == 'bfkm':
#         labels, centers = bfkm_clustering(X, sensitive, k)
#     elif method_name == 'fair_centroid':
#         labels, centers = fair_centroid_clustering(X, sensitive, k, max_iter=10, random_state=seed)
#     elif method_name == 'postprocessing_nfp':
#         labels, centers = postprocessing_clustering_nfp(X, sensitive, k, max_iter=15, random_state=seed)
#     elif method_name == 'postprocessing_gini':
#         labels, centers = postprocessing_clustering_gini(X, sensitive, k, max_iter=15, random_state=seed)
#     elif method_name == 'rawlsian':
#         labels, centers = rawlsian_kmeans(X, sensitive, k, n_runs=10, random_state=seed)
#     else:
#         raise ValueError(f"Unknown: {method_name}")
#     quality  = calculate_quality_metrics(X, labels)
#     fairness = {
#         'Balance':          calculate_balance(labels, sensitive),
#         'SPD':              calculate_spd(labels, sensitive),
#         'Disparate_Impact': calculate_disparate_impact(labels, sensitive),
#         'Entropy':          calculate_entropy_metric(labels, sensitive)
#     }
#     return quality, fairness

# FIXED (Reviewer 3, Comment 6): Calinski_Harabasz and BCSS removed from the
# aggregate utility set. Verified both algebraically and numerically that
# they are redundant with metrics already in this list under fixed k:
# BCSS = TSS - SSE(Inertia), where TSS is constant per dataset, so BCSS
# adds no information beyond Inertia; CHI = Variance_Ratio * (N-k)/(k-1),
# a constant scalar for fixed N and k, so CHI is a rescaled Variance_Ratio
# and produces identical rankings (confirmed via identical Friedman
# statistics on stored results before removal). Counting all as separate
# criteria would give repeated weight to the same underlying information.
UTIL_COLS = ['Inertia','Silhouette','Davies_Bouldin','Variance_Ratio']
# FIXED (Reviewer 3, Comment 2): 'Disparate_Impact' kept as the raw value
# (ideal = 1); 'DI_dist' = |DI-1| added as a separate ranking-oriented
# field (ideal = 0, lower always better), resolving the direction reversal
# the reviewer identified (e.g. DI=0.829 for K-Means vs. DI=0.417 for
# Fairlet was being interpreted as favoring Fairlet, although 0.829 is
# closer to parity -- |DI-1| now makes this comparison correct).
FAIR_COLS = ['Balance','SPD','Disparate_Impact','DI_dist','Entropy']

def run_single(X, sensitive, k, method_name, seed):
    np.random.seed(seed)
    if method_name == 'kmeans':
        labels, centers = standard_kmeans(X, k, random_state=seed)
    elif method_name == 'fairlet':
        labels, centers = fairlet_clustering(X, sensitive, k)
    elif method_name == 'bfkm':
        labels, centers = bfkm_clustering(X, sensitive, k)
    elif method_name == 'fair_centroid':
        labels, centers = fair_centroid_clustering(X, sensitive, k, max_iter=10, random_state=seed)
    elif method_name == 'postprocessing_nfp':
        # FIXED (Reviewer 3, Comment 4): max_iter 15 -> 30, matching Table 2's stated stopping criterion
        labels, centers = postprocessing_clustering_nfp(X, sensitive, k, max_iter=30, random_state=seed)
    elif method_name == 'postprocessing_gini':
        # FIXED (Reviewer 3, Comment 4): max_iter 15 -> 30, matching Table 2's stated stopping criterion
        labels, centers = postprocessing_clustering_gini(X, sensitive, k, max_iter=30, random_state=seed)
    elif method_name == 'rawlsian':
        # FIXED (Reviewer 3, Comment 4): n_runs 10 -> 30, matching Table 2's stated "30 k-means++ restarts"
        labels, centers = rawlsian_kmeans(X, sensitive, k, n_runs=30, random_state=seed)
    else:
        raise ValueError(f"Unknown: {method_name}")
    quality  = calculate_quality_metrics(X, labels)
    di_raw = calculate_disparate_impact(labels, sensitive)
    fairness = {
        'Balance':          calculate_balance(labels, sensitive),
        'SPD':              calculate_spd(labels, sensitive),
        'Disparate_Impact': di_raw,                  # FIXED: raw DI, ideal = 1 (Comment 2)
        'DI_dist':          abs(di_raw - 1.0),        # FIXED: new field, |DI-1|, ideal = 0, used for ranking (Comment 2)
        'Entropy':          calculate_entropy_metric(labels, sensitive)
    }
    return quality, fairness

# ── Main 30-run loop ──
all_results_30 = {}
methods = ['kmeans', 'fairlet', 'bfkm', 'fair_centroid',
           'postprocessing_nfp', 'postprocessing_gini', 'rawlsian']

print("\n" + "="*80)
print("RUNNING 30-SEED EXPERIMENTS — Strategy 1 (Majority Vote k)")
print("="*80)

total_start = time.perf_counter()

for name in dataset_names:
    X   = datasets[name]['X']
    sen = datasets[name]['sensitive']
    k   = approach2_results[name]['majority_vote_k']
    all_results_30[name] = {}

    print(f"\n{'='*60}")
    print(f"Dataset: {name.upper()} (majority k={k}, n={X.shape[0]})")
    print(f"{'='*60}")

    for method in methods:
        seeds  = SEEDS
        n_runs = len(seeds)

        # NEW: checkpoint / resume
        key = (name, method)
        if key in ckpt['done'] and ckpt['done'][key]['k'] == k:
            all_results_30[name][method] = ckpt['done'][key]
            _r = ckpt['done'][key]
            print(f"  {method.upper():25s} (k={k})  [restored from checkpoint, skipped]"
                  f"  Sil={_r['mean_quality']['Silhouette']:.3f}"
                  f"  Bal={_r['mean_fairness']['Balance']:.3f}")
            continue
        part = ckpt['partial'].get(key)
        # NEW: a partial CCF run made with the ORIGINAL (slow) CCF implementation is
        # restarted from seed 0. Its labels are identical, but mixing the two
        # implementations would make runtime_sec meaningless (Table 9).
        if part is not None and method == 'fair_centroid' and part.get('ccf_impl') != 'fast_v1':
            print(f"  {method.upper():25s} (k={k})  discarding {len(part['seeds_done'])} seed(s) "
                  f"run with the original CCF implementation, restarting")
            part = None
        if part is None or part['k'] != k:
            part = {'k': k, 'seeds_done': [], 'runs_q': [], 'runs_f': [], 'elapsed': 0.0}
            if method == 'fair_centroid':
                part['ccf_impl'] = 'fast_v1'
        elif part['seeds_done']:
            print(f"  {method.upper():25s} (k={k})  {len(part['seeds_done'])}/{n_runs} seeds already done")
        runs_q, runs_f = part['runs_q'], part['runs_f']

        for seed in seeds:
            if seed in part['seeds_done']:
                continue
            _t = time.perf_counter()
            q, f = run_single(X, sen, k, method, seed)
            runs_q.append(q); runs_f.append(f)
            part['seeds_done'].append(seed)
            part['elapsed'] += time.perf_counter() - _t     
            ckpt['partial'][key] = part
            save_ckpt(ckpt)                                   
        elapsed = part['elapsed']

        def agg(runs, cols):
            return (
                {c: float(np.mean([r[c] for r in runs])) for c in cols if c in runs[0]},
                {c: float(np.std( [r[c] for r in runs])) for c in cols if c in runs[0]}
            )
        mq, sq = agg(runs_q, UTIL_COLS)
        mf, sf = agg(runs_f, FAIR_COLS)

        all_results_30[name][method] = {
            'k':             k,
            'n_runs':        n_runs,
            'mean_quality':  mq,
            'std_quality':   sq,
            'mean_fairness': mf,
            'std_fairness':  sf,
            'all_quality':   runs_q,
            'all_fairness':  runs_f,
            'runtime_sec':   elapsed,
        }
        if method == 'fair_centroid':
            all_results_30[name][method]['ccf_impl'] = 'fast_v1'
        ckpt['done'][key] = all_results_30[name][method]
        ckpt['partial'].pop(key, None)
        save_ckpt(ckpt)

        print(f"  [{n_runs:2d} runs] {method.upper():25s}"
              f"  Sil={mq['Silhouette']:.3f}±{sq['Silhouette']:.3f}"
              f"  Bal={mf['Balance']:.3f}±{sf['Balance']:.3f}"
              f"  [{elapsed:.1f}s]")

total_elapsed = time.perf_counter() - total_start
print(f"\n✓ ALL 30-SEED EXPERIMENTS COMPLETED  Total: {total_elapsed/60:.1f} min")


RUNNING 30-SEED EXPERIMENTS — Strategy 1 (Majority Vote k)

Dataset: DIABETES_GENDER (majority k=4, n=99492)
  KMEANS                    (k=4)  [restored from checkpoint, skipped]  Sil=0.003  Bal=0.642
  FAIRLET                   (k=4)  [restored from checkpoint, skipped]  Sil=0.056  Bal=0.755
  BFKM                      (k=4)  [restored from checkpoint, skipped]  Sil=-0.109  Bal=0.562
  FAIR_CENTROID             (k=4)  [restored from checkpoint, skipped]  Sil=0.014  Bal=0.647
  POSTPROCESSING_NFP        (k=4)  [restored from checkpoint, skipped]  Sil=0.006  Bal=0.706
  POSTPROCESSING_GINI       (k=4)  [restored from checkpoint, skipped]  Sil=0.002  Bal=0.753
  RAWLSIAN                  (k=4)  [restored from checkpoint, skipped]  Sil=-0.009  Bal=0.760

Dataset: DIABETES_RACE (majority k=4, n=95309)
  KMEANS                    (k=4)  [restored from checkpoint, skipped]  Sil=0.014  Bal=0.175
  FAIRLET                   (k=4)  [restored from checkpoint, skipped]  Sil=0.003  Bal=0.250
  B

## SECTION 9: SAVE AND CREATE TABLES

## Save Results

In [25]:
n_fixed = 0
for ds in all_results_30:
    for method, r in all_results_30[ds].items():
        for f in r['all_fairness']:
            if f['Disparate_Impact'] == 0 and f['Balance'] != 0:
                f['Balance'] = 0.0; n_fixed += 1
        bal = [f['Balance'] for f in r['all_fairness']]
        r['mean_fairness']['Balance'] = float(np.mean(bal))
        r['std_fairness']['Balance']  = float(np.std(bal))
print(f"Balance corrected to Eq. (7) in {n_fixed} seed result(s)")

# Save 30-run results
with open('Strategy1_30runs_PartB.pkl', 'wb') as f:
    pickle.dump({
        'approach2_k': approach2_results,
        'clustering_results_30': all_results_30,
        'datasets': dataset_names,
        'seeds': SEEDS
    }, f)
print("✓ Saved: Strategy1_30runs_PartB.pkl")

# Build summary CSV
rows = []
for ds in all_results_30:
    for method in all_results_30[ds]:
        r = all_results_30[ds][method]
        row = {'Dataset': ds, 'Method': method,
               'k': r['k'], 'N_runs': r['n_runs']}
        for c in UTIL_COLS:
            if c in r['mean_quality']:
                row[f'{c}_mean'] = round(r['mean_quality'][c], 4)
                row[f'{c}_std']  = round(r['std_quality'][c],  4)
        for c in FAIR_COLS:
            if c in r['mean_fairness']:
                row[f'{c}_mean'] = round(r['mean_fairness'][c], 4)
                row[f'{c}_std']  = round(r['std_fairness'][c],  4)
        rows.append(row)

import pandas as pd
summary_df = pd.DataFrame(rows)
summary_df.to_csv('Strategy1_30runs_PartB_summary.csv', index=False)
print("✓ Saved: Strategy1_30runs_PartB_summary.csv")
print("\nPreview:")
print(summary_df[['Dataset','Method','k','N_runs',
                   'Silhouette_mean','Silhouette_std',
                   'Balance_mean','Balance_std']].head(14).to_string(index=False))

Balance corrected to Eq. (7) in 82 seed result(s)
✓ Saved: Strategy1_30runs_PartB.pkl
✓ Saved: Strategy1_30runs_PartB_summary.csv

Preview:
        Dataset              Method  k  N_runs  Silhouette_mean  Silhouette_std  Balance_mean  Balance_std
diabetes_gender              kmeans  4      30           0.0030          0.0387        0.3768       0.3300
diabetes_gender             fairlet  4      30           0.0559          0.1976        0.7547       0.0330
diabetes_gender                bfkm  4      30          -0.1090          0.0981        0.5374       0.2184
diabetes_gender       fair_centroid  4      30           0.0143          0.0292        0.3790       0.3308
diabetes_gender  postprocessing_nfp  4      30           0.0060          0.0381        0.6874       0.2449
diabetes_gender postprocessing_gini  4      30           0.0024          0.0382        0.7532       0.1057
diabetes_gender            rawlsian  4      30          -0.0095          0.0410        0.7388       0.1372
  di